In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/base_macro_mensal.csv", parse_dates=["data_ref"])
df.head()

,data_ref,ano,selic_meta,ipca_mensal,ipca_12m,dolar_ptax,ibc_br,juro_real_ex_post,dolar_var_12m,dolar_acima_media,ciclo_selic
0,2019-01-01,2019,6.5,0.32,3.78,3.7417,96.9521,2.6209,NaN,False,Estável
1,2019-02-01,2019,6.5,0.43,3.89,3.7236,96.3965,2.5123,NaN,False,Estável
2,2019-03-01,2019,6.5,0.75,4.58,3.8465,95.8017,1.8359,NaN,False,Estável
3,2019-04-01,2019,6.5,0.57,4.94,3.8962,95.7715,1.4866,NaN,False,Estável
4,2019-05-01,2019,6.5,0.13,4.66,4.0015,97.1148,1.7581,NaN,False,Estável


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data_ref           60 non-null     datetime64[us]
 1   ano                60 non-null     int64         
 2   selic_meta         60 non-null     float64       
 3   ipca_mensal        60 non-null     float64       
 4   ipca_12m           60 non-null     float64       
 5   dolar_ptax         60 non-null     float64       
 6   ibc_br             60 non-null     float64       
 7   juro_real_ex_post  60 non-null     float64       
 8   dolar_var_12m      48 non-null     float64       
 9   dolar_acima_media  60 non-null     bool          
 10  ciclo_selic        60 non-null     str           
dtypes: bool(1), datetime64[us](1), float64(7), int64(1), str(1)
memory usage: 5.3 KB


In [5]:
df.describe().round(2)

,data_ref,ano,selic_meta,ipca_mensal,ipca_12m,dolar_ptax,ibc_br,juro_real_ex_post,dolar_var_12m
count,60,60.00,60.00,60.00,60.00,60.00,60.00,60.00,48.00
mean,2021-06-16 06:24:00,2021.00,7.89,0.47,5.84,4.93,98.05,1.98,7.22
min,2019-01-01 00:00:00,2019.00,2.00,-0.68,1.88,3.72,82.96,-4.35,-14.46
25%,2020-03-24 06:00:00,2020.00,4.12,0.22,3.86,4.79,96.62,-1.04,-4.77
50%,2021-06-16 00:00:00,2021.00,6.50,0.49,4.66,5.14,97.71,1.27,-0.07
75%,2022-09-08 12:00:00,2022.00,12.88,0.83,8.13,5.30,100.81,6.31,18.63
max,2023-12-01 00:00:00,2023.00,13.75,1.62,12.13,5.65,104.90,10.27,41.03
std,NaN,1.43,4.50,0.46,2.94,0.57,4.07,4.21,16.30


In [6]:
# ===== Mês do menor e do maior juro real =====
pior = df.loc[df["juro_real_ex_post"].idxmin()]
melhor = df.loc[df["juro_real_ex_post"].idxmax()]
print("Menor juro real:", pior["data_ref"].strftime("%m/%Y"), round(pior["juro_real_ex_post"], 2))
print("Maior juro real:", melhor["data_ref"].strftime("%m/%Y"), round(melhor["juro_real_ex_post"], 2))

Menor juro real: 07/2021 -4.35
Maior juro real: 06/2023 10.27


In [7]:
# ===== Meses com juro real negativo =====
negativos = df[df["juro_real_ex_post"] < 0]
print("Meses com juro real negativo:", len(negativos))
negativos[["data_ref", "selic_meta", "ipca_12m", "juro_real_ex_post"]]

Meses com juro real negativo: 20


,data_ref,selic_meta,ipca_12m,juro_real_ex_post
18,2020-07-01,2.25,2.31,-0.0586
19,2020-08-01,2.00,2.44,-0.4295
20,2020-09-01,2.00,3.14,-1.1053
21,2020-10-01,2.00,3.92,-1.8476
22,2020-11-01,2.00,4.31,-2.2146
23,2020-12-01,2.00,4.52,-2.4110
24,2021-01-01,2.00,4.56,-2.4484
25,2021-02-01,2.00,5.20,-3.0418
26,2021-03-01,2.75,6.10,-3.1574
27,2021-04-01,2.75,6.76,-3.7561


In [8]:
# ===== Resumo por ano com groupby =====
resumo = df.groupby("ano").agg(
    meses_negativos=("juro_real_ex_post", lambda s: (s < 0).sum()),
    maior_juro_real=("juro_real_ex_post", "max"),
    menor_juro_real=("juro_real_ex_post", "min"),
    selic_media=("selic_meta", "mean"),
).round(2)
resumo

,meses_negativos,maior_juro_real,menor_juro_real,selic_media
ano,,,,
2019,0,3.18,0.18,5.96
2020,6,1.32,-2.41,2.81
2021,12,-0.74,-4.35,4.81
2022,2,7.52,-1.02,12.62
2023,0,10.27,6.82,13.25


In [12]:
# ===== Gráfico: Selic, IPCA 12M e juro real =====
import plotly.express as px

fig = px.line(df, x="data_ref", y=["selic_meta", "ipca_12m", "juro_real_ex_post"],
              labels={"value": "% a.a.", "data_ref": "", "variable": "Indicador"},
              title="Selic, inflação e juro real (2019-2023)")
fig.add_hline(y=0, line_color="black", line_width=1)
fig.show(renderer="notebook_connected")